### Trade flow imbalance indicators

In [17]:
import requests
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time

from concurrent.futures import ThreadPoolExecutor

In [28]:
def convert_to_strftime(t):
    dt = datetime.fromtimestamp(t / 1000, tz=timezone.utc)
    timestamp = dt.strftime('%Y-%m-%d %H:%M:%S.%f')
    return timestamp

def get_data(depth = 1000):
    order_book_depth = 1000
    r_long = requests.get("https://fapi.binance.com/fapi/v1/depth", params={"symbol": "BTCUSDT", "limit": order_book_depth})
    d_long = r_long.json()

    d_long['T'] = convert_to_strftime(d_long["T"])
    bids,asks = get_bid_asks(d_long)
    return bids,asks

def sample_df(d):
    return pd.DataFrame([{k: v[0] if isinstance(v, list) and v else v for k, v in d.items()}])

def get_bid_asks(frame):
    bids = np.array(frame["bids"], dtype=float)
    asks = np.array(frame["asks"], dtype=float)
    return bids, asks

def get_horizon_volumes(depth=1000):
    horizons_ms = [100, 250, 500, 1000, 2000]
    url = "https://fapi.binance.com/fapi/v1/depth"
    params = {"symbol": "BTCUSDT", "limit": depth}
    targets = [0] + horizons_ms
    session = requests.Session()

    with ThreadPoolExecutor(max_workers=len(targets)) as pool:
        # warm up connections so no timed request pays for the handshake
        list(pool.map(session.get, ["https://fapi.binance.com/fapi/v1/ping"] * len(targets)))

        futures = []
        t0 = time.perf_counter()
        for ms in targets:
            time.sleep(max(0, t0 + ms / 1000 - time.perf_counter()))
            futures.append(pool.submit(session.get, url, params=params))

        rows = []
        for ms, fut in zip(targets, futures):
            d = fut.result().json()
            if ms == 0:
                T0 = d["T"]
            rows.append({
                "target_ms": ms,
                "off_ms": (d["T"] - T0) - ms,     # actual gap from t=0 book minus target
                "ask_volume": sum(float(q) for _, q in d["asks"]),
                "bid_volume": sum(float(q) for _, q in d["bids"]),
            })

    return pd.DataFrame(rows)

def get_tfi(symbol="BTCUSDT"):
    horizons_ms = [100, 250, 500, 1000, 2000]
    base = "https://fapi.binance.com/fapi/v1"
    session = requests.Session()

    t = session.get(f"{base}/time").json()["serverTime"]   # t on Binance's clock
    time.sleep(0.5)                                         # let the latest trades get published
    start = t - max(horizons_ms)

    trades = {}
    while True:
        batch = session.get(f"{base}/aggTrades", params={
            "symbol": symbol, "startTime": start, "endTime": t, "limit": 1000}).json()
        for tr in batch:
            trades[tr["a"]] = tr
        if len(batch) < 1000 or batch[-1]["T"] == start:
            break
        start = batch[-1]["T"]

    rows = []
    for d in horizons_ms:
        window = [tr for tr in trades.values() if tr["T"] >= t - d]     # [t - δ, t]
        buy = sum(float(tr["p"]) * float(tr["q"]) for tr in window if not tr["m"])
        sell = sum(float(tr["p"]) * float(tr["q"]) for tr in window if tr["m"])
        rows.append({"delta_ms": d, "buy_usd": buy, "sell_usd": sell, "TFI": buy - sell})
    return pd.DataFrame(rows)

In [31]:
df = get_tfi()

In [32]:
df

,delta_ms,buy_usd,sell_usd,TFI
0,100,0.0000,233901.2313,-233901.2313
1,250,748.3518,233901.2313,-233152.8795
2,500,20621.2258,287699.3427,-267078.1169
3,1000,241218.4411,321707.6927,-80489.2516
4,2000,242299.3924,375089.9927,-132790.6003
